# Word2Vec: noun-only lemmatization

This notebook reads the shared training `words` column and lemmatizes tokens tagged `N*` on a private copy. Token count, order, digits, negation, and degree words such as `better/best/worse/worst` stay as they are. It does not overwrite the shared text columns, and it does not use WebMD, phrases, or stop-word filtering. The vocabulary is fit on train. Validation and test only apply the frozen rules.

| Parameter | Default |
|---|---:|
| vector_size | 300 |
| window_size | 5 |
| min_count | 5 |
| max_iter | 10 |
| step_size | 0.025 |
| seed | 42 |
| num_partitions | 1 |
| max_sentence_length | 1000 |
| pooling | meanmax |

`meanmax` concatenates the mean and the per-coordinate maximum, so the vector is **600**-dimensional. The mean includes out-of-vocabulary tokens; the maximum does not. An empty or fully unknown review becomes a zero vector of that length. `p01` is a 300-dimensional mean, also after noun-only lemmatization.

Use the README environment: Python 3.12, PySpark 4.2, and Java 21. The delivered model also needs **NLTK 3.9.3**, **WordNet 3.0**, and the English POS resource. Run this once in the driver and worker environment:

```bash
python -m pip install nltk==3.9.3
python -m nltk.downloader averaged_perceptron_tagger_eng wordnet
```

NLTK uses its default search path or `NLTK_DATA`. The code does not download resources. The model stores a resource fingerprint and checks the version and SHA256 at prediction time. Java and Hadoop are configured outside this notebook.

## 1. Run settings

- `REBUILD=False` loads the saved split. Set it to `True` on a fresh checkout or after cleaning, text features, the split, or encoding change. That overwrites the shared split.
- `TRAIN=False` loads the delivered noun-only model and rebuilds its vector cache. `TRAIN=True` refits on train, ordered by `uniqueID`, and replaces the model at `model_name`. `REBUILD` does not control training.
- `word2vec_config={}` uses the module defaults. A custom config also needs `TRAIN=True`. Prediction always reads the saved config.
- Use a separate `model_name` for other experiments. Noun-only lemmatization can still be ambiguous, for example `doses` to `dos`. There is no dataset-specific exception list.

Loading an older surface-form model raises an error. Update the delivered model, or retrain with `TRAIN=True`. Restart the kernel after changing the module, Java, or memory.

In [ ]:
import hashlib
import json
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

ROOT = Path.cwd()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "src").is_dir() and (candidate / "project.ipynb").is_file():
        ROOT = candidate
        break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from pyspark.sql import SparkSession
from src.dataOI import DataOI
from src.paths import EMBEDDING_MODELS_DIR, EMBEDDING_DIR, SPLIT_DIR

from src.seed import SEED
REBUILD = False
TRAIN = False
embedding_name = "word2vec"
model_name = "word2vec"
word2vec_config = {}

if not TRAIN and word2vec_config:
    raise ValueError("Custom config requires TRAIN=True; prediction uses the saved model config.")

spark = (SparkSession.builder
         .master("local[4]")
         .appName("Word2VecPipeline")
         .config("spark.driver.memory", "4g")
         .config("spark.sql.shuffle.partitions", "8")
         .config("spark.sql.parquet.columnarReaderBatchSize", "512")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
data_oi = DataOI()

## 2. Shared inputs

Use the shared cleaning, text features, `seed=42` hash split, and the encoder fit on train. Word2Vec reads `words`. Its noun lemmatization does not change those cached columns.

A git pull does not refresh local Parquet. Each run records the input row count and a content fingerprint. An older experiment score is not a rerun on the current shared code.

In [ ]:
from src.cleaning import data_cleaning
from src.text_features import text_feature_engineering
from src.encoder import encoder
from src.split import split, weight

if REBUILD:
    df_raw = data_oi.load_raw_data()
    df_cleaned = data_cleaning(df_raw)
    df_cleaned.cache().count()
    df_engineered = text_feature_engineering(df_cleaned, True)
    df_engineered.cache().count()
    df_train, df_valid, df_test = split(df_engineered, SEED)
    df_train = weight(df_train)
    df_train, df_valid, df_test = encoder(df_train, df_valid, df_test)
    data_oi.save_split_data(df_train, df_valid, df_test)
    data_oi.unpersist(df_cleaned, df_engineered)
    del df_raw, df_cleaned, df_engineered
else:
    df_train, df_valid, df_test = data_oi.load_split_data()


## 3. Train or load

The registry key stays `word2vec`. `model_name` chooses the directory. Training refits and saves the config. Prediction loads that directory and does not fit on validation or test. The saved config must declare `noun_only`.

The input SHA256 is one digest of relative paths and file contents. The run record is `run_record.json` in the local cache and is not committed.

In [ ]:
from src.embeddings import EmbeddingModel

embedding_model = EmbeddingModel(df_train, df_valid, df_test)
if TRAIN:
    print(f"Training noun-only Word2Vec: {model_name}")
    EmbeddingModel.train_registry[embedding_name](
        embedding_model, model_name=model_name, config=word2vec_config,
    )
else:
    print(f"Reusing saved noun-only Word2Vec: {model_name}")

saved_config = json.loads(
    (EMBEDDING_MODELS_DIR / model_name / "embedding_config.json").read_text(encoding="utf-8")
)
if (saved_config.get("format_version") != 2
        or saved_config.get("normalization", {}).get("policy") != "noun_only"):
    raise ValueError("This notebook requires the noun-only model. Update the delivered model or set TRAIN=True with the current module.")
print(json.dumps(saved_config, ensure_ascii=False, indent=2))


def input_fingerprint(directory):
    digest = hashlib.sha256()
    files = sorted(directory.rglob("*.parquet"))
    for path in files:
        digest.update(path.relative_to(directory).as_posix().encode("utf-8") + b"\0")
        with path.open("rb") as stream:
            for block in iter(lambda: stream.read(1024 * 1024), b""):
                digest.update(block)
    return {"path": str(directory), "parquet_files": len(files), "sha256": digest.hexdigest()}


run_record = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "code_commit": subprocess.check_output(["git", "-C", str(ROOT), "rev-parse", "HEAD"], text=True).strip(),
    "embedding": model_name,
    "trained_this_run": TRAIN,
    "split_rebuilt": REBUILD,
    "split_seed": SEED,
    "saved_model_config": saved_config,
    "inputs": {
        name: {**input_fingerprint(SPLIT_DIR / f"{name}.parquet"), "rows": frame.count()}
        for name, frame in zip(["train", "valid", "test"], [df_train, df_valid, df_test])
    },
}

## 4. Write the three vector frames

Prediction uses the saved lemmatizer and pooling config and returns train, validation, and test. The base class keeps IDs, labels, the training weight, and the encoded features. `embedding` is a Spark ML vector.

`DataOI` writes `data/processed/embeddings/<model_name>/` and replaces an older cache of the same name. Downstream classifiers need a new fit. This notebook does not run other analytics models.

In [ ]:
print(f"Generating embeddings: {model_name}")
train_embedding, valid_embedding, test_embedding = (
    EmbeddingModel.predict_registry[embedding_name](embedding_model, model_name)
)
data_oi.save_embedding_data(
    model_name, train_embedding, valid_embedding, test_embedding,
)
run_record["output"] = {
    "path": str(EMBEDDING_DIR / model_name),
    "dimension": saved_config["output_dimension"],
    "rows": {
        name: frame.count()
        for name, frame in zip(["train", "valid", "test"], [train_embedding, valid_embedding, test_embedding])
    },
}
assert all(run_record["output"]["rows"][name] == record["rows"]
           for name, record in run_record["inputs"].items())
record_path = EMBEDDING_DIR / model_name / "run_record.json"
record_path.write_text(json.dumps(run_record, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print(f"Saved {saved_config['output_dimension']}D embeddings to {EMBEDDING_DIR / model_name}")
print(f"Run record: {record_path}")
train_embedding.printSchema()
train_embedding.select("uniqueID", "rating", "embedding").show(3, truncate=80)

## Selected model

The delivered model is `word2vec_noun_only_20261111_154957`, from `selectlemma_20261111_154957/noun_only`. The experiment code was `6e9ae80` and the input cache came from `46b1a03`. It was trained on 77,800 training reviews and 6,702,823 tokens. 295,995 tokens were lemmatized, and the vocabulary after `minCount` has 13,812 entries. NLTK POS and WordNet are frozen resources. Validation and test labels were not used.

These validation scores use the same historical split and the same unweighted `lr_baseline`. They are not a rerun on the current shared pipeline.

| Variant | Validation accuracy | Weighted F1 | MAE | RMSE |
|---|---:|---:|---:|---:|
| Noun only (selected) | 40.4145% | 0.353624 | 1.740379 | 2.934713 |
| Surface form | 40.2261% | 0.350837 | 1.770970 | 2.976644 |
| Full lemmatization | 39.5891% | 0.341728 | 1.833498 | 3.051485 |

On 1,976 validation reviews with true ratings 4–7, the correct counts were 126, 108, and 93. Noun-only gained 18 correct predictions over the surface form. The exploratory paired p-value was about 0.142. All three logistic models hit the 100-iteration limit. This was one seed and a fixed budget. Test predictions were saved and not scored.

See `embedding_config.json` in the model directory for the source and input summary. Each later run records its fingerprint in the local `run_record.json`. Load the three frames with `data_oi.load_embedding_data(model_name)`. Clear notebook outputs before committing, and do not commit the generated cache.